# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [6]:
# Drop rows with any missing values
df_clean = spaceship.dropna()

print("Original shape:", spaceship.shape)
print("Clean shape:", df_clean.shape)
print("Rows removed:", spaceship.shape[0] - df_clean.shape[0])

Original shape: (8693, 14)
Clean shape: (6606, 14)
Rows removed: 2087


In [7]:

df_clean["Deck"] = df_clean["Cabin"].str.split("/").str[0]

print(df_clean["Deck"].unique())

['B' 'F' 'A' 'G' 'E' 'C' 'D' 'T']


C:\Users\Utilizador\AppData\Local\Temp\ipykernel_11060\309098031.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["Deck"] = df_clean["Cabin"].str.split("/").str[0]


In [8]:
df_clean.drop(columns=["PassengerId", "Name"], inplace=True)

C:\Users\Utilizador\AppData\Local\Temp\ipykernel_11060\2685270498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(columns=["PassengerId", "Name"], inplace=True)


In [9]:
df_encoded = pd.get_dummies(df_clean, drop_first=True)

print(df_encoded.head())

    Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  Transported  \
0  39.0          0.0        0.0           0.0     0.0     0.0        False   
1  24.0        109.0        9.0          25.0   549.0    44.0         True   
2  58.0         43.0     3576.0           0.0  6715.0    49.0        False   
3  33.0          0.0     1283.0         371.0  3329.0   193.0        False   
4  16.0        303.0       70.0         151.0   565.0     2.0         True   

   HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  ...  \
0               True            False           False  ...   
1              False            False           False  ...   
2               True            False           False  ...   
3               True            False           False  ...   
4              False            False           False  ...   

   Destination_PSO J318.5-22  Destination_TRAPPIST-1e  VIP_True  Deck_B  \
0                      False                     True     False    True   
1       

In [12]:
from sklearn.preprocessing import StandardScaler


X = df_encoded.drop("Transported", axis=1)
y = df_encoded["Transported"]


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Perform Train Test Split**

In [13]:
from sklearn.model_selection import train_test_split


X = df_encoded.drop("Transported", axis=1)
y = df_encoded["Transported"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5284, 5323)
X_test: (1322, 5323)
y_train: (5284,)
y_test: (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [14]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report


base_model = DecisionTreeClassifier(random_state=42)

# -------------------------
# 1. Bagging
# -------------------------
bagging_model = BaggingClassifier(
    estimator=base_model,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,  
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_train, y_train)
y_pred_bagging = bagging_model.predict(X_test)

print("Bagging Accuracy:", accuracy_score(y_test, y_pred_bagging))
print("\nBagging Classification Report:\n")
print(classification_report(y_test, y_pred_bagging))


# -------------------------
# 2. Pasting
# -------------------------
pasting_model = BaggingClassifier(
    estimator=base_model,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=False,  
    random_state=42,
    n_jobs=-1
)

pasting_model.fit(X_train, y_train)
y_pred_pasting = pasting_model.predict(X_test)

print("Pasting Accuracy:", accuracy_score(y_test, y_pred_pasting))
print("\nPasting Classification Report:\n")
print(classification_report(y_test, y_pred_pasting))

Bagging Accuracy: 0.7813918305597579

Bagging Classification Report:

              precision    recall  f1-score   support

       False       0.79      0.76      0.78       656
        True       0.77      0.80      0.79       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Pasting Accuracy: 0.791981845688351

Pasting Classification Report:

              precision    recall  f1-score   support

       False       0.80      0.78      0.79       656
        True       0.79      0.80      0.80       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



- Random Forests

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)


rf_model.fit(X_train, y_train)


y_pred_rf = rf_model.predict(X_test)


print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.7859304084720121

Classification Report:

              precision    recall  f1-score   support

       False       0.78      0.79      0.78       656
        True       0.79      0.79      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322


Confusion Matrix:

[[516 140]
 [143 523]]


- Gradient Boosting

In [16]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)


print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_gb))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.7798789712556732

Classification Report:

              precision    recall  f1-score   support

       False       0.81      0.72      0.77       656
        True       0.75      0.83      0.79       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322


Confusion Matrix:

[[475 181]
 [110 556]]


- Adaptive Boosting

In [17]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada_model.fit(X_train, y_train)


y_pred_ada = ada_model.predict(X_test)

print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_ada))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_ada))

AdaBoost Accuracy: 0.7299546142208775

Classification Report:

              precision    recall  f1-score   support

       False       0.68      0.86      0.76       656
        True       0.81      0.60      0.69       666

    accuracy                           0.73      1322
   macro avg       0.75      0.73      0.73      1322
weighted avg       0.75      0.73      0.73      1322


Confusion Matrix:

[[563  93]
 [264 402]]


Which model is the best and why?

In [19]:
# Random Forest showed overall better results with almost 79% accuracy